In [1]:
import sys
import os

# Append root project folder to sys.path so src imports work seamlessly
sys.path.append(os.path.abspath(".."))

import json
from src.models.prediction_pipeline.pipeline import StoryPointPredictionPipeline

/home/chris2003/Uni/MasterTUM/IDP/predictive-code-complexity/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load sample training & validation datasets
def load_json_dataset(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    stories = [item["story"] for item in data]
    points = [float(item["points"]) for item in data]
    return stories, points

train_stories, train_points = load_json_dataset("../data/train.json")
val_stories, val_points = load_json_dataset("../data/val.json")

print(f"Loaded {len(train_stories)} training samples and {len(val_stories)} validation samples.")

In [ ]:
# Adjust parameters directly without running automated search
HYPERPARAMS = {
    "model_name": "gpt2",
    "lr": 2e-4,
    "weight_decay": 0.01,
    "dropout": 0.15,
    "hidden_dim": 256,
    "batch_size": 8,
    "freeze_strategy": "partial",  # Options: 'full' or 'partial'
    "unfreeze_layers_from": 8,     # Keep bottom 0-7 layers frozen
    "epochs": 15,
    "patience": 5
}

In [ ]:
# Initialize pipeline
pipeline = StoryPointPredictionPipeline(model_name=HYPERPARAMS["model_name"])

# Start training (TQDM progress bar renders interactively below)
metrics = pipeline.fit(
    train_stories=train_stories,
    train_points=train_points,
    val_stories=val_stories,
    val_points=val_points,
    lr=HYPERPARAMS["lr"],
    weight_decay=HYPERPARAMS["weight_decay"],
    dropout=HYPERPARAMS["dropout"],
    hidden_dim=HYPERPARAMS["hidden_dim"],
    batch_size=HYPERPARAMS["batch_size"],
    freeze_strategy=HYPERPARAMS["freeze_strategy"],
    unfreeze_layers_from=HYPERPARAMS["unfreeze_layers_from"],
    epochs=HYPERPARAMS["epochs"],
    patience=HYPERPARAMS["patience"]
)

print(f"Training Complete! Final Best Val MAE: {metrics['val_mae']:.4f}")

In [ ]:
# Verify model predictions inside notebook
test_story = "As a developer, I want to integrate OAuth2 authentication for Google and GitHub."
predicted_points = pipeline.predict([test_story])

print(f"Story: {test_story}")
print(f"Predicted Story Points: {predicted_points[0]}")

In [ ]:
# Export model weights for CLI inference
WEIGHTS_PATH = "../weights/story_point_model.pt"
pipeline.save_weights(WEIGHTS_PATH)